# Titanic Machine Learning Challenge
**The Challenge**

The sinking of the Titanic is one of the most infamous shipwrecks in history.

On April 15, 1912, during her maiden voyage, the widely considered “unsinkable” RMS Titanic sank after colliding with an iceberg. Unfortunately, there weren’t enough lifeboats for everyone onboard, resulting in the death of 1502 out of 2224 passengers and crew.

While there was some element of luck involved in surviving, it seems some groups of people were more likely to survive than others.

In this challenge, we ask you to build a predictive model that answers the question: “what sorts of people were more likely to survive?” using passenger data (ie name, age, gender, socio-economic class, etc).

# Titanic Survival Prediction in R with Decision Trees

We will be using the "rpart" (Recursive Partitioning and Regression Trees) library in R to building decision trees to classify passengers in the "test.csv" file as "Survived" Yes (1) or No (0) based on the following


* Survived 0 = No, 1 = Yes
* Pclass	Ticket class	1 = 1st, 2 = 2nd, 3 = 3rd
* Sex	Sex
* Age	Age in years
* SibSp	# of siblings / spouses aboard the Titanic
* Parch	# of parents / children aboard the Titanic
* Ticket	Ticket number
* Fare	Passenger fare
* Cabin	Cabin number
* Embarked	Port of Embarkation	C = Cherbourg, Q = Queenstown, S = Southampton


In a nutshell rpart "rpart" splits the data into subsets based on the values of the input variables the result being a tree like model which is nice to interpret and visualize, bascially a series of splits down the tree making deceisions along the way.

In [101]:
#install.packages("rpart")
library(rpart)
library(rpart.plot)
#install.packages("pROC")  # if not already installed
library(pROC)

# Data Preparation and Cleaning

First, we read in the training data, "train.csv" and inspect the structure of the dataframe, data types and number of variables.

We can see we have 891 passengers covering 12 variables.

There are 5 categorical variables "Name", "Sex", "Ticket", "Cabin", "Embarked".

There are 7 numerical variables "PassengerId", "Survived", "Pclass", "SibSp", "Parch" and "Fare".

In [102]:
#Read in titanic training data
training <- read.csv("/kaggle/input/titanic/train.csv")

# Inspect dataframe structure and datatypes
str(training)

numerical_cols <- c("PassengerId", "Survived", "Pclass", "SibSp", "Parch", "Fare")
categorical_cols <- c("Name", "Sex", "Ticket", "Cabin", "Embarked")

'data.frame':	891 obs. of  12 variables:
 $ PassengerId: int  1 2 3 4 5 6 7 8 9 10 ...
 $ Survived   : int  0 1 1 1 0 0 0 0 1 1 ...
 $ Pclass     : int  3 1 3 1 3 3 1 3 3 2 ...
 $ Name       : chr  "Braund, Mr. Owen Harris" "Cumings, Mrs. John Bradley (Florence Briggs Thayer)" "Heikkinen, Miss. Laina" "Futrelle, Mrs. Jacques Heath (Lily May Peel)" ...
 $ Sex        : chr  "male" "female" "female" "female" ...
 $ Age        : num  22 38 26 35 35 NA 54 2 27 14 ...
 $ SibSp      : int  1 1 0 1 0 0 0 3 0 1 ...
 $ Parch      : int  0 0 0 0 0 0 0 1 2 0 ...
 $ Ticket     : chr  "A/5 21171" "PC 17599" "STON/O2. 3101282" "113803" ...
 $ Fare       : num  7.25 71.28 7.92 53.1 8.05 ...
 $ Cabin      : chr  "" "C85" "" "C123" ...
 $ Embarked   : chr  "S" "C" "S" "S" ...


# Inspect dataframe structure and datatypes
head(training)

In [103]:
# Summary statistics of training dataframe numerical columns
summary(training[numerical_cols])

  PassengerId       Survived          Pclass          SibSp      
 Min.   :  1.0   Min.   :0.0000   Min.   :1.000   Min.   :0.000  
 1st Qu.:223.5   1st Qu.:0.0000   1st Qu.:2.000   1st Qu.:0.000  
 Median :446.0   Median :0.0000   Median :3.000   Median :0.000  
 Mean   :446.0   Mean   :0.3838   Mean   :2.309   Mean   :0.523  
 3rd Qu.:668.5   3rd Qu.:1.0000   3rd Qu.:3.000   3rd Qu.:1.000  
 Max.   :891.0   Max.   :1.0000   Max.   :3.000   Max.   :8.000  
     Parch             Fare       
 Min.   :0.0000   Min.   :  0.00  
 1st Qu.:0.0000   1st Qu.:  7.91  
 Median :0.0000   Median : 14.45  
 Mean   :0.3816   Mean   : 32.20  
 3rd Qu.:0.0000   3rd Qu.: 31.00  
 Max.   :6.0000   Max.   :512.33  

In [104]:
# frequency data of the categorial columns bar PassengerId and Fare
lapply(training[c("Survived", "Pclass", "Sex", "Cabin", "Embarked","SibSp", "Parch" )], table)

$Survived

  0   1 
549 342 

$Pclass

  1   2   3 
216 184 491 

$Sex

female   male 
   314    577 

$Cabin

                            A10             A14             A16             A19 
            687               1               1               1               1 
            A20             A23             A24             A26             A31 
              1               1               1               1               1 
            A32             A34             A36              A5              A6 
              1               1               1               1               1 
             A7            B101            B102             B18             B19 
              1               1               1               2               1 
            B20             B22             B28              B3             B30 
              2               2               2               1               1 
            B35             B37             B38             B39              B4

Looks like we have 687 values in "Cabin" and 2 in "Embarked" that look like empty strings, let's confirm this and check elsewhere.

In [105]:
# Then find where the empty strings are in training
empty_counts <- sapply(training, function(x) if (is.character(x)) sum(x == "", na.rm = TRUE) else 0)

# View columns with any empty strings
output <- paste(names(empty_counts[empty_counts >0]), "(", empty_counts[empty_counts>0], ")", collapse = ', ')
                       
paste("Columns with empty strings are:", output) 

[1] "Columns with empty strings are: Cabin ( 687 ), Embarked ( 2 )"

We can now change the empty strings to NA as a default way to address the empty strings.

In [106]:
# Make "" empty strings in Cabin and Embarked NA
training[c("Cabin", "Embarked")] <- lapply(training[c("Cabin", "Embarked")], function(x) ifelse(x == "", NA, x))

# Check the "" empty strings are now NA       
head(training[is.na(training$Cabin), ])
head(training[is.na(training$Embarked), ])

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
,<int>,<int>,<int>,<chr>,<chr>,<dbl>,<int>,<int>,<chr>,<dbl>,<chr>,<chr>
1,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.2500,NA,S
3,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.9250,NA,S
5,5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.0500,NA,S
6,6,0,3,"Moran, Mr. James",male,NA,0,0,330877,8.4583,NA,Q
8,8,0,3,"Palsson, Master. Gosta Leonard",male,2,3,1,349909,21.0750,NA,S
9,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27,0,2,347742,11.1333,NA,S


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
,<int>,<int>,<int>,<chr>,<chr>,<dbl>,<int>,<int>,<chr>,<dbl>,<chr>,<chr>
62,62,1,1,"Icard, Miss. Amelie",female,38,0,0,113572,80,B28,NA
830,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62,0,0,113572,80,B28,NA


We will check for NA values in the dataframe.

In [107]:
# Check for missing values - there are 177 missing ages in the data
print(colSums(is.na(training)))

PassengerId    Survived      Pclass        Name         Sex         Age 
          0           0           0           0           0         177 
      SibSp       Parch      Ticket        Fare       Cabin    Embarked 
          0           0           0           0         687           2 


# Predictor Variables Exlcuded

**Drop Cabin** Given 687 missing values in Cabin and the nature of the data we will exclude it from the model at this stage.

**Drop Ticket** is a letter and a number will little meaning, so will be dropped from the model, 

**Drop PassengerId** is a generic ID number with no relevance to prediction

**Drop Name** whilst it might be interesting to attempt to model, at this stage it will be dropped

**Drop Embarked** pre modelling suggests did not improve predictions from the model on the test data



# Predictor Variables Included


1. **Pclass**
2. **Sex**
3. **Age**
4. **SibSp**
5. **Parch**
6. **Fare**


# Missing Data Imputation

For the missing values in Age, try a few different imputation methods


* None = rpart will deal with it
* Median = Assign median age for NA values in Age
* GroupWise = - This replaces NA values in x (the Age values for that group) with the median of the non-missing values in that group based on Pclass and Sex
* Mice = MICE, or Multiple Imputation by Chained Equations offers a robust way to preserve relationships between variables and reduce bias compared to simpler methods like mean or median imputation.

Let's make a list of these methods and the code used to impute the "Age" variable so we can loop through them later in the modelling stage to compare model accuracy.


In [108]:
#Create a list of the imputation methods for Age
imputation_methods <- list(
    none = function(df) df,

    # NA -> Median
    median = function(df) {
        df$Age[is.na(df$Age)] <- median(df$Age, na.rm = TRUE)
        df
    },
    
    # NA -> Median grouped by class and sex
    groupwise = function(df) {
        df$Age <- ave(df$Age, df$Pclass, df$Sex, FUN = function(x) {
            x[is.na(x)] <- median(x, na.rm = TRUE)
            return(x)
        })
        df
    },

    # NA ->
    mice = function(df) {
        library(mice)
        imp <- mice(df, m = 1, printFlag = FALSE)
        return(complete(imp, 1))
    }
)  

Build a function to create a model based on training data, evaluate accuracy and store the results for comparison including test data prediction rates and prepare submission files for the competition.

In [109]:
# Function to create a model, based on training data, evaluate accuracy and store
# as well as comparing test data outcomes and prepare submission files
create_model <- function(training, impute_method = imputation_methods){
    
    #Assign training to df so it is not modified data each time
    df <- training
    
    # Apply the selected imputation method from the list above
    df <- imputation_methods[[impute_method]](df)
    
    # Create the model - will output a decision tree based on predicting        survived
    # using Pclass, Sex, Age, SibSp, Parch, Fare
    model <- rpart(
        formula = Survived ~ Pclass + Sex + Age + SibSp + Parch + Fare,
        data = df,
        method = "class"
    )
    
    # Plot the Decision Tree
    # Open JPEG device
    jpeg("my_decision_tree.jpg", width = 800, height = 600)

    # Plot the decision tree
    rpart.plot(model)

    # Close the device to save the file
    dev.off()
    
    #Use the model to predict the survival of the training data passengers
    df$Prediction <- predict(
        object = model,
        newdata = df,
        type = "class"
    )

    # Create confusion matrix and accuracy metrics
    conf_matrix <- table(df$Prediction,df$Survived)
    accuracy <-  sum(diag(conf_matrix)/sum(conf_matrix))
    sensitivity <- conf_matrix[2,2]/sum(conf_matrix[,2])
    specificity <- conf_matrix[1,1]/sum(conf_matrix[,1])    
    
    # read in test data
    test <- read.csv("/kaggle/input/titanic/test.csv")
    
    # Use the model to predict if passengers survived, store in column "Survived"
    test$Survived<- predict(
        object = model,
        newdata = test,
        type = "class"
    )
    
    #Create a dataframe to return with the survival predictions, method and accuracy
    evaluation_model <- data.frame(
        SurvivalPrediction = names(table(test$Survived)),
        Freq = as.vector(table(test$Survived)),
        Method = impute_method,
        Accuracy = accuracy,
        Sensitivity = conf_matrix[2,2]/sum(conf_matrix[,2]),
        Specificity = conf_matrix[1,1]/sum(conf_matrix[,1])
    )
    
    # Create a submission dataframe of the PassengerId from the test data and the prediction
    submission <- data.frame(PassengerId = test$PassengerId, Survived = test$Survived)
    
    # Create a submission file name with the impute method in the name
    submission_filename <- paste("/kaggle/working/submission", impute_method, ".csv", sep = "")
    
    #write file to CSV
    write.csv(submission, submission_filename, row.names = FALSE)
    
    #Return the results dataframe
    return(evaluation_model)
}

# The None Strategy actually scored the best accuracy in the Kaggle Titanic Competition, as in it predicted who survived the best on the test data at .78468

In [110]:
# Create an empty results dataframe
results <- data.frame()


# For each method, create the model using the training data, and assign results
for (method_name in names(imputation_methods)){
    model_results <- create_model(training, method_name)
    results <- rbind(results, model_results)
}

Warning message:
“Number of logged events: 5”


In [111]:
#Print out results
results

write.csv(results, "/kaggle/working/results.csv")

SurvivalPrediction,Freq,Method,Accuracy,Sensitivity,Specificity
<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>
0,284,none,0.8417508,0.6871345,0.9380692
1,134,none,0.8417508,0.6871345,0.9380692
0,284,median,0.8417508,0.6929825,0.9344262
1,134,median,0.8417508,0.6929825,0.9344262
0,291,groupwise,0.8439955,0.6871345,0.9417122
1,127,groupwise,0.8439955,0.6871345,0.9417122
0,297,mice,0.8383838,0.6608187,0.9489982
1,121,mice,0.8383838,0.6608187,0.9489982
